# TP 1 — Du texte brut au premier modèle bigramme

**Objectif :** comprendre comment un corpus devient un ensemble de prédictions supervisées, puis entraîner une table qui prédit le caractère suivant.

1. Télécharger et explorer Tiny Shakespeare
2. Construire `encode` et `decode`
3. Former contextes, cibles et mini-batchs
4. Programmer le modèle bigramme
5. Générer, entraîner, puis comparer

**Livrable :** ce notebook exécuté de haut en bas, deux générations (avant / après entraînement) et les réponses aux questions d'interprétation.

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)
print(torch.__version__)

2.11.0+cu130


## Étape 1 — Télécharger et inspecter le corpus

Le modèle n'apprend pas directement Shakespeare : il reçoit une longue suite de caractères.
Relevez : la longueur du corpus, les retours à la ligne, la ponctuation, les majuscules et minuscules.

In [2]:
from urllib.request import urlretrieve
import os

url = ("https://raw.githubusercontent.com/karpathy/"
       "char-rnn/master/data/tinyshakespeare/input.txt")
urlretrieve(url, "input.txt")
print(os.path.abspath("input.txt"))

with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

print(len(text))
print(text[:1000])

/content/input.txt
1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, n

In [3]:
# Quelques observations chiffrées
print("Nombre de lignes      :", text.count("\n"))
print("Majuscules            :", sum(c.isupper() for c in text))
print("Minuscules            :", sum(c.islower() for c in text))
print("Ponctuation présente  :", sorted(set(c for c in text if not c.isalnum() and not c.isspace())))

Nombre de lignes      : 40000
Majuscules            : 88754
Minuscules            : 762324
Ponctuation présente  : ['!', '$', '&', "'", ',', '-', '.', ':', ';', '?']


### Questions d'interprétation

**Quelle est ici l'unité d'un token ?**
*Réponse :* un caractère (lettre, chiffre, espace, retour à la ligne, signe de ponctuation).

**Pourquoi préciser `encoding="utf-8"` ?**
*Réponse :* pour que les octets du fichier soient décodés de la même façon quelle que soit la machine (l'encodage par défaut dépend du système, ex. cp1252 sous Windows) ; sinon certains caractères pourraient être mal lus ou provoquer une erreur.

**Le fichier est-il déjà séparé en exemples ?**
*Réponse :* non, c'est un seul texte continu. Les exemples (contexte → cible) seront découpés nous-mêmes en tirant des fenêtres dans cette suite.

## Étape 2 — Passer des caractères aux entiers

PyTorch manipule des nombres : on construit un vocabulaire, puis deux dictionnaires parfaitement inverses.

In [4]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print("".join(chars))
print("vocab_size =", vocab_size)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join([itos[i] for i in ids])

message = "hii there"
print(encode(message))
print(decode(encode(message)))
assert decode(encode(message)) == message


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
vocab_size = 65
[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


### Questions d'interprétation

**Pourquoi trier `chars` ?**
*Réponse :* `set` n'a pas d'ordre garanti ; trier rend le vocabulaire (donc les identifiants) déterministe et reproductible d'une exécution à l'autre.

**Que garantit le test `decode(encode(message)) == message` ?**
*Réponse :* que `stoi` et `itos` sont bien inverses l'un de l'autre : l'encodage est sans perte, on peut toujours retrouver le texte à partir des entiers.

*Remarque :* un identifiant n'a pas de sens par lui-même ; il désigne seulement une ligne dans une table.

## Étape 3 — Fabriquer les prédictions attendues

Une même séquence de longueur `T+1` fournit une entrée `x` et une cible `y` décalée d'une position : chaque token doit prédire le suivant.

In [5]:
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:100])

n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

block_size = 8
x = train_data[:block_size]
y = train_data[1:block_size + 1]

for t in range(block_size):
    context = x[:t + 1]
    target = y[t]
    print(context, "->", target, f"   ({decode(context.tolist())!r} -> {decode([target.item()])!r})")

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])
tensor([18]) -> tensor(47)    ('F' -> 'i')
tensor([18, 47]) -> tensor(56)    ('Fi' -> 'r')
tensor([18, 47, 56]) -> tensor(57)    ('Fir' -> 's')
tensor([18, 47, 56, 57]) -> tensor(58)    ('Firs' -> 't')
tensor([18, 47, 56, 57, 58]) -> tensor(1)    ('First' -> ' ')
tensor([18, 47, 56, 57, 58,  1]) -> tensor(15)    ('First ' -> 'C')
tensor([18, 47, 56, 57, 58,  1, 15]) -> tensor(47)    ('First C' -> 'i')
tensor([18, 47, 56, 57, 58,  1, 15, 47]) -> tensor(58)    ('First Ci' -> 't')


### Questions d'interprétation

1. **Combien de prédictions produit ce bloc ?** *Réponse :* `block_size = 8` prédictions, une par position (contextes de longueur 1 à 8), à partir de 9 caractères.
2. **Pourquoi ne pas mélanger train et validation ?** *Réponse :* la validation doit mesurer la capacité à généraliser sur du texte jamais vu. Si ces données servaient à l'entraînement, la loss de validation serait optimiste et ne détecterait plus le surapprentissage.
3. **Pour t = 3, quels sont le contexte et la cible ?** *Réponse :* contexte = `x[:4]`, soit « Firs », et cible = `y[3]`, soit « t ».

## Étape 4 — Échantillonner un mini-batch

Un mini-batch regroupe `B` séquences indépendantes de longueur `T`. Toutes les prédictions sont calculées en parallèle.

In [6]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(split):
    source = train_data if split == "train" else val_data
    starts = torch.randint(len(source) - block_size, (batch_size,))
    x = torch.stack([source[i : i + block_size] for i in starts])
    y = torch.stack([source[i + 1 : i + block_size + 1] for i in starts])
    return x, y

xb, yb = get_batch("train")
print(xb.shape, yb.shape)   # (B, T) = (4, 8)
print(xb)
print(yb)

torch.Size([4, 8]) torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])


Une ligne = une séquence indépendante. Une colonne = une position temporelle.

### Questions d'interprétation

1. **Pourquoi retirer `block_size` à la borne de tirage ?** *Réponse :* pour que `source[i+1 : i+block_size+1]` reste entièrement dans les données ; sans cela, une fenêtre tirée trop près de la fin serait plus courte et `torch.stack` échouerait.
2. **Que changerait B = 32 ?** *Réponse :* 32 séquences par batch → 32×8 = 256 prédictions par pas, un gradient moins bruité, plus de calcul et de mémoire par pas. Le modèle ne voit pas plus loin dans le passé.
3. **Que changerait T = 64 ?** *Réponse :* des fenêtres plus longues : 64 prédictions par séquence et des contextes jusqu'à 64 caractères. Pour le bigramme, cela n'apporte rien (il ne regarde que le dernier caractère), mais ce sera utile pour l'attention.

## Étape 5 — Une table produit les logits

Chaque caractère choisit directement une ligne contenant un score pour chacun des `C` caractères possibles : « après ce caractère, quels caractères sont plausibles ? »

In [7]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)
        # idx: (B,T) -> logits: (B, T, C) avec C = vocab_size

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx : (B, T) indices du contexte courant
        for _ in range(max_new_tokens):
            logits, _ = self(idx)               # (B, T, C)
            logits = logits[:, -1, :]           # (B, C) : dernier pas de temps
            probs = F.softmax(logits, dim=-1)   # (B, C)
            idx_next = torch.multinomial(probs, num_samples=1)  # (B, 1)
            idx = torch.cat((idx, idx_next), dim=1)             # (B, T+1)
        return idx

torch.manual_seed(1337)
model = BigramLanguageModel(vocab_size)
logits, loss = model(xb, yb)
print(logits.shape)   # (B*T, C) car des cibles ont été fournies
print(loss)
print("loss attendue pour une prédiction uniforme : -ln(1/65) =",
      torch.log(torch.tensor(float(vocab_size))).item())

torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)
loss attendue pour une prédiction uniforme : -ln(1/65) = 4.174387454986572


### Questions d'interprétation

**Pourquoi C × C ici ?**
*Réponse :* la table a une ligne par caractère d'entrée (C lignes) et chaque ligne contient un score pour chacun des C caractères de sortie possibles. Elle sert à la fois d'embedding et de couche de sortie.

**Où retrouvera-t-on d_emb et la couche `lm_head` dans le TP 4 ?**
*Réponse :* la table sera séparée en deux : `token_embedding_table` de taille `(vocab_size, n_embd)` (identité du caractère dans un espace de dimension d_emb) puis, après les blocs, `lm_head = nn.Linear(n_embd, vocab_size)` qui ramène chaque vecteur aux C logits du vocabulaire.

## Étapes 6 et 7 — Générer, puis apprendre

On génère d'abord avec les poids aléatoires, on entraîne, puis on relance exactement la même procédure.

In [8]:
# Génération AVANT entraînement
torch.manual_seed(42)
context = torch.zeros((1, 1), dtype=torch.long)   # caractère 0 = '\n'
print(decode(model.generate(context, max_new_tokens=300)[0].tolist()))


vixDYq,SfqHxdf?aN:Cju &yAU:wGZ3fBOSVjKDnqbObHZ.p EVyZe?Ya!a-Ugd
lvmhi-X!qfrdRj?YWKXcm;oDG&uCgRl3QJGS.uwF!PWTqH'u
JevviA3O
KmUIBjm:'&vv''&yKLUzlhZcObvWM?bHOgkprgnq n?q-I,SFPJnqWWSGfO,StuQL!uUiP!Biq,nbdcQW?ERFUAHPRT;.yuEXEOpHN!
pZhSPlnW:gCuFd
.cOVM:PmnT Yfk.?m qag:wFI&sJjqvbOjXbQqxcmayCIl,bTEYd?Y-s bJ


In [9]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

batch_size = 32
for step in range(100):
    xb, yb = get_batch("train")
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if step % 20 == 0 or step == 99:
        print(step, loss.item())

0 4.6666059494018555
20 4.714624404907227
40 4.705251693725586
60 4.690428733825684
80 4.640739917755127
99 4.58889627456665


In [10]:
# Génération APRÈS 100 étapes
torch.manual_seed(42)
print(decode(model.generate(context, max_new_tokens=300)[0].tolist()))


vixDYq,SfqHxdf?aN:Cju &yAU:wGZ3fBOSVjKDnqbObHZ.p EVyZe?Ya!a-Ugd
lvi-gUz,qfrdRj?YWKXcm;oDG&uCgRl3QJGS.uwF!PWTqH'u
JevviA3O
KmUIBjm:'&vv''&yKLUzlhZcObvWM?bHO:C-rgnq n?q-I,SFPJnqWWSGfO,StuQL!uUiP!Biq,nbdcQW?ERFUAHPRT;.yuEXEOpHN!
pZhSPlnW:gCuFd
.cOVM:PmnT Yfk.?m qag:wFI&sJjqvbOjXbQqxcmayCIl,bTEYd?Y-s bJ


In [11]:
# On poursuit jusqu'à 1 000 étapes au total
for step in range(100, 1000):
    xb, yb = get_batch("train")
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if step % 100 == 0 or step == 999:
        print(step, loss.item())

torch.manual_seed(42)
print(decode(model.generate(context, max_new_tokens=300)[0].tolist()))

100 4.547152996063232
200 4.431727886199951
300 4.402612209320068
400 4.220245838165283
500 4.242920398712158
600 4.160045623779297
700 4.0275983810424805
800 3.8163249492645264
900 3.876861333847046
999 3.672058343887329

visDYq,Sfa
SttRay:Cju &yeh:w'daP,pSVjKDnqbOnmZ.p EVyZe?Ya!a-Ugd
lvi-gUz,qfrdRj?YWKXcmaoDG&uCgRl3QJGS.uwF!
WTqHAu
JevviA3O
KmUIBjuCUzoa!L&yeLUzlhoctR:qM?bofak-rgng n?q-I,SFPJnqWWhlyc,SthothuUirOVz:,nodcQW?ERGUAHPRT; nuEXEOpHN!
pphSPlmWIKDW:d
.ce?wN Mny Yfk.?m aag wFI&sJjqvbOjXbQqxcmayCIs, TEYd?Y-s bJ


In [12]:
# Variante (facultatif) : learning rate plus élevé et plus d'étapes
model2 = BigramLanguageModel(vocab_size)
opt2 = torch.optim.AdamW(model2.parameters(), lr=1e-2)
for step in range(5000):
    xb, yb = get_batch("train")
    _, loss = model2(xb, yb)
    opt2.zero_grad(set_to_none=True)
    loss.backward()
    opt2.step()
print("loss finale :", loss.item())
torch.manual_seed(42)
print(decode(model2.generate(context, max_new_tokens=300)[0].tolist()))

loss finale : 2.282677412033081


K:
YOLIfe
Thand nd f my ch s nd, onoutndy nd fe t y e tar heed lamaton!

SS:
I be thall k tgnd heere, t!
Whithu
Lereie t

NUSIOfoondat nde lflercthin wboughareng n d I
S:
An if Got, thothunir his, buccoun codith h huru ONINomy he lloll ut cearen: mnd bef th ang wen


Thesitobutuck,
CIS: whedeals br


## Bilan du TP 1

**Pourquoi ne garde-t-on que le dernier pas de temps pendant la génération ?**
Le modèle calcule des logits pour toutes les positions, mais seul le dernier donne la distribution du caractère *suivant* le contexte ; les autres positions prédisent des caractères déjà connus.

**Pourquoi aplatit-on B × T pendant la loss ?**
`F.cross_entropy` attend des logits de forme `(N, C)` et des cibles `(N,)`. Chaque position de chaque séquence est une prédiction indépendante : on en fait donc une liste de N = B·T exemples.

**Comparaison avant/après (résultats obtenus).**
- **Avant entraînement :** loss ≈ 4,88, au-dessus de ln(65) ≈ 4,17 car les logits initiaux ne sont pas uniformes ; le texte est une suite aléatoire de caractères.
- **Après 100 étapes (lr = 1e-3) :** loss ≈ 4,59. Le texte généré est presque identique au précédent (même graine aléatoire, poids à peine modifiés).
- **Après 1 000 étapes :** loss ≈ 3,67. Le texte reste très bruité, mais quelques espaces et retours à la ligne apparaissent davantage. Avec lr = 1e-3, la table apprend lentement.
- **Variante (lr = 1e-2, 5 000 étapes) :** loss ≈ 2,28 sur le dernier batch. On voit clairement des noms de personnages en majuscules suivis de « : », des retours à la ligne et des syllabes plausibles, mais pas de vrais mots : le bigramme ne voit qu'un seul caractère de contexte, ce qui limite sa loss autour de 2,4–2,5 en moyenne.

**Ce que l'entraînement modifie réellement :** uniquement les valeurs de la table C × C, c'est-à-dire les scores « caractère suivant » associés à chaque caractère.

**Comment des logits deviennent des probabilités :** par Softmax, p_j = exp(z_j) / Σ_k exp(z_k).